# 第11回 画像処理：値とコードを一段ずつ読む

画素の位置を示し、判定・近傍の集計・符号付きエッジ・領域特徴・位置ごとの照合を追います。

**開始：** 既存venvのカーネルを選び、初回は変更せず「カーネル再起動→すべて実行」。次に一節ずつ、予想してからShift+Enterで実行します。同梱Pythonのimportは不要で、外部通信も行いません。

**読み方：** 関数の定義は計算方法に名前を付けるセルです。続く短い実行セルが入力を渡します。表・図の表示コードは折り畳み指定です。表示は丸めますが計算には元の値を使います。

**完了の目印：** 中央の平均25/9・中央値2、ラプラシアン−28、領域面積9、テンプレートの行1列2でSSD=0。


In [ ]:
import sys
print('使用中のPython:',sys.executable)
print('版:',sys.version.split()[0])


In [ ]:
import argparse


In [ ]:
from copy import deepcopy


In [ ]:
import json


In [ ]:
import math


In [ ]:
IMAGE = [[0,0,0,0,0],[0,2,2,2,0],[0,2,9,2,0],[0,2,2,2,0],[0,0,0,0,0]]


In [ ]:
SCENE = [[0,0,0,0,0],[0,0,1,2,0],[0,0,3,4,0],[0,0,0,0,0]]


In [ ]:
TEMPLATE = [[1,2],[3,4]]


In [ ]:
POOL_IMAGE = [[1,2,3,4],[5,6,7,8],[9,10,11,12],[13,14,15,16]]


In [ ]:
KERNELS = {'laplacian':[[0,1,0],[1,-4,1],[0,1,0]],
           'sobel_x':[[-1,0,1],[-2,0,2],[-1,0,1]]}


### 図表の準備（変更不要）

表示用の補助コードです。HTML/SVGを覚える必要はありません。折り畳み表示はNotebook環境によって異なります。


In [ ]:
from html import escape


In [ ]:
from IPython.display import HTML, display


In [ ]:
def learning_format(value):
    if isinstance(value, float):
        return f'{value:.6g}'
    if isinstance(value, (list, tuple)):
        return '['+', '.join(learning_format(item) for item in value)+']'
    if isinstance(value, dict):
        return '{'+', '.join(str(key)+': '+learning_format(item) for key,item in value.items())+'}'
    return str(value)


In [ ]:
def learning_table(title, columns, rows):
    header = ''.join('<th>'+escape(str(c))+'</th>' for c in columns)
    body = ''
    for row in rows:
        body += '<tr>'+''.join('<td>'+escape(learning_format(x))+'</td>' for x in row)+'</tr>'
    return '<div class="scroll"><table><caption>'+escape(title)+'</caption><tr>'+header+'</tr>'+body+'</table></div>'


In [ ]:
def learning_grid(title, matrix):
    rows = [(i,*row) for i,row in enumerate(matrix)]
    return learning_table(title,['行↓ 列→',*range(len(matrix[0]))],rows)


In [ ]:
def learning_plot(title, points):
    # 実計算の値だけを座標へ変換。点の間の値は補間して表示しない。
    xs, ys = [p[0] for p in points], [p[1] for p in points]
    xmin,xmax,ymin,ymax = min(xs),max(xs),min(0,min(ys)),max(ys)
    sx = lambda x: 100+(x-xmin)/max(1e-12,xmax-xmin)*500
    sy = lambda y: 240-(y-ymin)/max(1e-12,ymax-ymin)*190
    marks = ''
    for value in (ymin,(ymin+ymax)/2,ymax):
        marks += f'<path d="M100 {sy(value)} H600" stroke="#aaa"/><text x="0" y="{sy(value)+5}">{value:.4g}</text>'
    for x,y in points:
        marks += f'<circle cx="{sx(x)}" cy="{sy(y)}" r="4" fill="#155e75"><title>{x}: {y}</title></circle>'
    marks += f'<text x="100" y="275">{xmin}</text><text x="580" y="275">{xmax}</text>'
    return '<svg viewBox="0 0 700 310" role="img" aria-label="'+escape(title)+'"><text x="100" y="22">'+escape(title)+'</text>'+marks+'</svg>'


In [ ]:
def learning_show(report, reset=None):
    style = '<style>.learning{font:17px/1.65 sans-serif;color:#172b42;background:white;padding:12px}.learning .scroll{overflow-x:auto}.learning table{border-collapse:collapse;margin:14px 0}.learning td,.learning th{border:1px solid #8394a5;padding:6px 12px;white-space:normal}.learning caption{text-align:left;font-weight:bold}.learning svg{max-width:100%;height:auto}.learning .boards{display:flex;gap:20px;flex-wrap:wrap}.learning .result{border-left:4px solid #155e75;padding:10px}</style>'
    content = '<h3>'+escape(report['title'])+'</h3><p>入力：'+escape(learning_format(report['input']))+'</p>'
    content += learning_table('使う値 → 操作 → 結果',report['columns'],report['rows'])
    for title,columns,rows in report.get('tables',[]):
        content += '<details><summary>詳しい計算：'+escape(title)+'</summary>'+learning_table(title,columns,rows)+'</details>'
    content += '<div class="boards">'+''.join(learning_grid(title,values) for title,values in report.get('matrices',[]))+'</div>'
    for title,points in report.get('plots',[]):
        content += learning_plot(title,points)
    for label in ('result','comparison','explanation','reset'):
        name={'result':'結果','comparison':'一つ変えた比較','explanation':'読み取り方','reset':'戻す'}[label]
        content += '<p class="result">'+name+'：'+escape(reset if label=='reset' and reset is not None else learning_format(report[label]))+'</p>'
    display(HTML(style+'<div class="learning">'+content+'</div>'))


## 1. 位置ごとの判定を読む

**今いる段階：** 画素→比較→0/1

**予想：** しきい値と等しい2は残るか

**操作：** example_thresholdを1から2へ変更し、同じセルを実行します。

**結果の目印：** 2は0になり、9だけが残る

**戻す：** example_threshold=1に戻して再実行します。

**説明する：** 比較演算子>の意味


### 計算の定義：`finite`

引数から結果を求める処理。次の実行セルで、具体的な入力を渡して確認します。


In [ ]:
def finite(x, low=-1e6, high=1e6):
    if isinstance(x, bool) or not isinstance(x,(int,float)) or not math.isfinite(x) or not low <= x <= high:
        raise ValueError(f'有限数 {low}〜{high} を指定してください')
    return x


### 計算の定義：`matrix`

引数から結果を求める処理。次の実行セルで、具体的な入力を渡して確認します。


In [ ]:
def matrix(a):
    if not isinstance(a, list) or not 1 <= len(a) <= 16 or (not isinstance(a[0], list)) or (not 1 <= len(a[0]) <= 16):
        raise ValueError('行列は1〜16行・列の二重リストです')
    w = len(a[0])
    for row in a:
        if not isinstance(row, list) or len(row) != w:
            raise ValueError('各行の列数をそろえてください')
        for x in row:
            finite(x)
    return (len(a), w)


### 計算の定義：`binary`

引数から結果を求める処理。次の実行セルで、具体的な入力を渡して確認します。


In [ ]:
def binary(image, threshold=1):
    matrix(image)
    finite(threshold, 0, 255)
    return [[int(x > threshold) for x in row] for row in image]


### 計算の定義：`features`

1の個数が面積。1の位置の行・列をそれぞれ平均すると重心。


In [ ]:
def features(mask):
    """1の個数が面積。1の位置の行・列をそれぞれ平均すると重心。"""
    height, width = matrix(mask)
    if any(x not in (0,1) for row in mask for x in row):
        raise ValueError('二値マスクの値は0か1です')
    area = sum(sum(row) for row in mask)
    row_terms, col_terms = [], []
    for row in range(height):
        for col in range(width):
            row_terms.append(row*mask[row][col])
            col_terms.append(col*mask[row][col])
    centroid = None if area == 0 else [sum(row_terms)/area,sum(col_terms)/area]
    rows = [sum(row) for row in mask]
    columns = [sum(mask[row][col] for row in range(height)) for col in range(width)]
    return {'area':area,'centroid':centroid,'rows':rows,'columns':columns}


### 計算の定義：`lesson_binary`

引数から結果を求める処理。次の実行セルで、具体的な入力を渡して確認します。


In [ ]:
def lesson_binary(threshold=1):
    mask = binary(IMAGE,threshold)
    rows = [(r,c,value,f'{value}>{threshold}',mask[r][c]) for r,line in enumerate(IMAGE) for c,value in enumerate(line)]
    return {'title':'画素の位置と二値化','question':'しきい値と同じ値は1になるか','input':f'5×5画像、しきい値{threshold}',
            'columns':['行','列','画素値','判定','0/1'],'rows':rows,
            'matrices':[('入力画素',IMAGE),('判定後',mask)],'result':f"1の画素数={features(mask)['area']}",
            'comparison':f"しきい値2では面積={features(binary(IMAGE,2))['area']}",
            'explanation':'判定は「より大きい」。同じ値は0。二値化は画像の位置を変えず値を0/1にする。',
            'reset':'threshold=1で再実行。'}


In [ ]:
example_threshold = 1
program_observation_binary = lesson_binary(example_threshold)
learning_show(program_observation_binary, reset='example_threshold=1に戻して再実行します。')


## 2. 9個の値を1個へ集約

**今いる段階：** 窓→9値→1出力

**予想：** 外れ値9はどう影響するか

**操作：** example_methodを'mean'から'median'へ変更し、同じセルを実行します。

**結果の目印：** 平均25/9、中央値2

**戻す：** example_method='mean'に戻して再実行します。

**説明する：** 平均と中央値の計算方法の違い


### 計算の定義：`window`

引数から結果を求める処理。次の実行セルで、具体的な入力を渡して確認します。


In [ ]:
def window(image,row,col,h,w):
    return [line[col:col+w] for line in image[row:row+h]]


### 計算の定義：`filtering`

3×3窓の値・積・結果を、出力位置と対応させて計算する。


In [ ]:
def filtering(image, method='mean', *, record=None):
    """3×3窓の値・積・結果を、出力位置と対応させて計算する。"""
    height, width = matrix(image)
    if height < 3 or width < 3:
        raise ValueError('3×3以上の画像を指定してください')
    if method not in ['mean','median','range',*KERNELS]:
        raise ValueError('未知のフィルタです')
    output = []
    for row in range(height-2):
        output_row = []
        for col in range(width-2):
            patch = window(image,row,col,3,3)
            values = [pixel for line in patch for pixel in line]
            products = []
            if method == 'mean':
                value = sum(values)/9
            elif method == 'median':
                value = sorted(values)[4]
            elif method == 'range':
                value = max(values)-min(values)
            else:
                for i in range(3):
                    for j in range(3):
                        products.append(image[row+i][col+j]*KERNELS[method][i][j])
                value = sum(products)
            output_row.append(value)
            if record is not None:
                record({'row':row,'col':col,'patch':patch,'values':values,'products':products,'value':value})
        output.append(output_row)
    return output


### 計算の定義：`lesson_filter`

引数から結果を求める処理。次の実行セルで、具体的な入力を渡して確認します。


In [ ]:
def lesson_filter(method='mean'):
    records = []
    result = filtering(IMAGE,method,record=records.append)
    center = records[4]
    rows = [(i//3,i%3,value) for i,value in enumerate(center['values'])]
    return {'title':'近傍処理：中央の3×3から1値へ','question':'中央の9という外れ値で平均と中央値はどう違うか',
            'input':f'中央窓の開始[1,1]、処理={method}','columns':['窓内行','窓内列','値'],'rows':rows,
            'tables':[('同じ9値で比較',['合計','個数','平均','並べ替えた値','中央値'],[(sum(center['values']),9,sum(center['values'])/9,str(sorted(center['values'])),sorted(center['values'])[4])])],
            'matrices':[('入力',IMAGE),('切り出した3×3',center['patch']),('出力3×3',result)],
            'result':f"出力の行1・列1={center['value']}", 'comparison':f"中央値処理の中央={filtering(IMAGE,'median')[1][1]}",
            'explanation':'窓を1マスずつ動かすため5×5から3×3になる。中央値は合計ではなく、9個を並べた5番目。',
            'reset':"method='mean'で再実行。"}


In [ ]:
example_method = 'mean'
program_observation_filter = lesson_filter(example_method)
learning_show(program_observation_filter, reset="example_method='mean'に戻して再実行します。")


## 3. 重みを掛けて符号を読む

**今いる段階：** 画素×重み→積和

**予想：** 明暗反転で合計はどうなるか

**操作：** example_invertをFalseからTrueへ変更し、同じセルを実行します。

**結果の目印：** 中央−28が＋28へ

**戻す：** example_invert=Falseに戻して再実行します。

**説明する：** 負の出力を0にしない理由


### 計算の定義：`lesson_edge`

引数から結果を求める処理。次の実行セルで、具体的な入力を渡して確認します。


In [ ]:
def lesson_edge(invert=False):
    image = [[9-x for x in row] for row in IMAGE] if invert else deepcopy(IMAGE)
    records = []
    output = filtering(image,'laplacian',record=records.append)
    center = records[4]
    rows = [(i//3,i%3,center['values'][i],KERNELS['laplacian'][i//3][i%3],product) for i,product in enumerate(center['products'])]
    return {'title':'符号付きのエッジ：積を足す','question':'明暗を逆にすると符号はどうなるか',
            'input':f'中央3×3、明暗反転={invert}','columns':['行','列','画素','重み','積'],'rows':rows,
            'matrices':[('入力',image),('ラプラシアンの重み',KERNELS['laplacian']),('符号を残した出力',output)],
            'result':f"中央の合計={center['value']}", 'comparison':f"元の画像の中央={filtering(IMAGE,'laplacian')[1][1]}",
            'explanation':'負の値は失敗ではなく明暗変化の向きを含む。表示で絶対値や0〜255に変換する操作とは分ける。',
            'reset':'invert=Falseで再実行。'}


In [ ]:
example_invert = False
program_observation_edge = lesson_edge(example_invert)
learning_show(program_observation_edge, reset='example_invert=Falseに戻して再実行します。')


## 4. 1の位置から重心へ

**今いる段階：** 二値領域→面積→重心

**予想：** 面積0の重心はどこか

**操作：** example_thresholdを1から9へ変更し、同じセルを実行します。

**結果の目印：** 面積0、重心None

**戻す：** example_threshold=1に戻して再実行します。

**説明する：** 0で割れない場合の意味


### 計算の定義：`lesson_features`

引数から結果を求める処理。次の実行セルで、具体的な入力を渡して確認します。


In [ ]:
def lesson_features(threshold=1):
    mask = binary(IMAGE,threshold)
    result = features(mask)
    rows = [(r,c,mask[r][c],r*mask[r][c],c*mask[r][c]) for r in range(5) for c in range(5) if mask[r][c]]
    if not rows:rows=[('—','—',0,0,0)]
    return {'title':'領域の特徴：面積と重心','question':'1が一つもない場合、重心をどう扱うか',
            'input':f'しきい値{threshold}の二値マスク','columns':['行','列','1なら数える','行×値','列×値'],'rows':rows,
            'matrices':[('領域',mask)], 'result':result,
            'comparison':f'しきい値9なら{features(binary(IMAGE,9))}',
            'explanation':'面積は1の合計、重心は行の重み付き和÷面積と列の重み付き和÷面積。面積0なら重心はNone（未定義）。',
            'reset':'threshold=1で再実行。'}


In [ ]:
example_threshold = 1
program_observation_features = lesson_features(example_threshold)
learning_show(program_observation_features, reset='example_threshold=1に戻して再実行します。')


## 5. 各位置の4画素とスコア

**今いる段階：** 位置→窓→差→スコア表

**予想：** 一致する窓のSSDはいくつか

**操作：** example_positionを6から0へ変更し、同じセルを実行します。

**結果の目印：** 位置6は行1列2、SSD=0

**戻す：** example_position=6に戻して再実行します。

**説明する：** スコアの大小と未定義を区別する


### 計算の定義：`score`

引数から結果を求める処理。次の実行セルで、具体的な入力を渡して確認します。


In [ ]:
def score(patch, template, method='ssd'):
    if matrix(patch) != matrix(template):
        raise ValueError('同じ形のパッチとテンプレートを指定してください')
    a = [x for row in patch for x in row]
    b = [x for row in template for x in row]
    if method == 'ssd':
        return sum(((x - y) ** 2 for x, y in zip(a, b)))
    if method != 'zncc':
        raise ValueError('評価方法はssdかznccです')
    ma = sum(a) / len(a)
    mb = sum(b) / len(b)
    a = [x - ma for x in a]
    b = [x - mb for x in b]
    denominator = math.sqrt(sum((x * x for x in a)) * sum((x * x for x in b)))
    if denominator == 0:
        return None
    return sum((x * y for x, y in zip(a, b))) / denominator


### 計算の定義：`match`

引数から結果を求める処理。次の実行セルで、具体的な入力を渡して確認します。


In [ ]:
def match(image, template, method='ssd'):
    h, w = matrix(image)
    th, tw = matrix(template)
    if th > h or tw > w:
        raise ValueError('テンプレートは画像以下の大きさにしてください')
    if method not in ('ssd', 'zncc'):
        raise ValueError('評価方法はssdかznccです')
    scores = [[score(window(image, r, c, th, tw), template, method) for c in range(w - tw + 1)] for r in range(h - th + 1)]
    candidates = [(s, r, c) for r, row in enumerate(scores) for c, s in enumerate(row) if s is not None]
    if not candidates:
        return {'scores': scores, 'best': None, 'ties': []}
    value = (min if method == 'ssd' else max)((s for s, r, c in candidates))
    ties = [[r, c] for s, r, c in candidates if abs(s - value) <= 1e-12]
    return {'scores': scores, 'best': {'row': ties[0][0], 'col': ties[0][1], 'score': value}, 'ties': ties}


### 計算の定義：`tone`

引数から結果を求める処理。次の実行セルで、具体的な入力を渡して確認します。


In [ ]:
def tone(image, offset=0):
    matrix(image)
    finite(offset, -255, 255)
    for row in image:
        for x in row:
            finite(x, 0, 255)
    return [[min(255, max(0, x + offset)) for x in row] for row in image]


### 計算の定義：`lesson_match`

引数から結果を求める処理。次の実行セルで、具体的な入力を渡して確認します。


In [ ]:
def lesson_match(position=6):
    width = len(SCENE[0])-len(TEMPLATE[0])+1
    row,col = divmod(position,width)
    patch = window(SCENE,row,col,2,2)
    matrix(patch)
    ssd = match(SCENE,TEMPLATE,'ssd')
    zncc = match(SCENE,TEMPLATE,'zncc')
    rows = [(i,j,patch[i][j],TEMPLATE[i][j],patch[i][j]-TEMPLATE[i][j],(patch[i][j]-TEMPLATE[i][j])**2) for i in range(2) for j in range(2)]
    all_scores = [(r,c,value,zncc['scores'][r][c]) for r,line in enumerate(ssd['scores']) for c,value in enumerate(line)]
    return {'title':'テンプレート：位置ごとに比較した4画素とスコア','question':'一致する位置のSSDはいくつか',
            'input':f'開始位置[{row},{col}]、出力スコア表の同じ位置','columns':['窓内行','窓内列','画像','テンプレート','差','二乗差'],'rows':rows,
            'tables':[('全12位置の比較',['行','列','SSD（小さい方）','ZNCC（大きい方）'],all_scores)],
            'matrices':[('全体画像',SCENE),('選択中の窓',patch),('テンプレート',TEMPLATE),('SSDの位置対応',ssd['scores'])],
            'result':f"選択窓のSSD={score(patch,TEMPLATE)}、最良={ssd['best']}",
            'comparison':f"同じテンプレートに明るさ10を足すとSSD={score(tone(TEMPLATE,10),TEMPLATE)}、ZNCC={score(tone(TEMPLATE,10),TEMPLATE,'zncc')}",
            'explanation':'SSDは差の二乗和。ZNCCは平均を引き長さで割るため一様な明るさ変化には違う応答をする。定数窓では分母0となりNone。同点は全件表示し最初の位置を代表にする。',
            'reset':'position=6で再実行。位置は0〜11。'}


In [ ]:
example_position = 6
program_observation_match = lesson_match(example_position)
learning_show(program_observation_match, reset='example_position=6に戻して再実行します。')


## 基準結果と既存の変更実験

ここまでの図表を、従来の基準出力と照合します。以下の実験も入力のコピーを使います。定義を変更した場合は再起動して全セル実行してください。


### 計算の定義：`pool`

引数から結果を求める処理。次の実行セルで、具体的な入力を渡して確認します。


In [ ]:
def pool(image):
    h, w = matrix(image)
    if h % 2 or w % 2:
        raise ValueError('2×2プーリングでは行・列を偶数にしてください')
    return [[max((image[r + i][c + j] for i in range(2) for j in range(2))) for c in range(0, w, 2)] for r in range(0, h, 2)]


### 計算の定義：`rotate`

引数から結果を求める処理。次の実行セルで、具体的な入力を渡して確認します。


In [ ]:
def rotate(image):
    h,w=matrix(image)
    return [[image[h-1-r][c] for r in range(h)] for c in range(w)]


### 計算の定義：`rounded`

引数から結果を求める処理。次の実行セルで、具体的な入力を渡して確認します。


In [ ]:
def rounded(value):
    if isinstance(value,float): return round(value,6)
    if isinstance(value,list): return [rounded(x) for x in value]
    if isinstance(value,dict): return {k:rounded(v) for k,v in value.items()}
    return value


### 計算の定義：`demo`

引数から結果を求める処理。次の実行セルで、具体的な入力を渡して確認します。


In [ ]:
def demo():
    return rounded({'binary':binary(IMAGE),'mean':filtering(IMAGE),'median':filtering(IMAGE,'median'),
      'laplacian':filtering(IMAGE,'laplacian'),'sobel_x':filtering(IMAGE,'sobel_x'),'features':features(binary(IMAGE)),
      'pool':pool(POOL_IMAGE),'ssd':match(SCENE,TEMPLATE),'zncc':match(SCENE,TEMPLATE,'zncc'),
      'brightness_pair':{'ssd':score(tone(TEMPLATE,10),TEMPLATE),'zncc':score(tone(TEMPLATE,10),TEMPLATE,'zncc')},
      'constant_zncc':score([[2,2],[2,2]],TEMPLATE,'zncc')})


In [ ]:
baseline_result = demo()
print(json.dumps(baseline_result, ensure_ascii=False, indent=2))


## 二値化のしきい値

問い：t=2で前景は何画素か。

予想：1のままか、減るかを予想する。

想定結果：前景1画素、重心(2,2)。

説明する問い：画素値2が前景に入らない理由を説明する。


In [ ]:
trial_threshold = binary(deepcopy(IMAGE), 2)
print(features(trial_threshold))


## 孤立画素の変更

問い：中心9を2へ戻すと平均と中央値はどうなるか。

予想：どちらが大きく変わるかを予想する。

想定結果：平均2、中央値2。基準の平均は25/9。

説明する問い：出力の中心以外も変わる理由を説明する。


In [ ]:
trial_image = deepcopy(IMAGE)
trial_image[2][2] = 2
trial_smooth = filtering(trial_image)
print(trial_smooth[1][1], filtering(trial_image, "median")[1][1])


## 明暗反転と符号

問い：9−画素値の画像ではラプラシアンの中心はどうなるか。

予想：符号と大きさを予想する。

想定結果：−28から+28。

説明する問い：絶対値だけを見ると何が区別できなくなるか。


In [ ]:
trial_inverse = [[9-x for x in row] for row in IMAGE]
trial_edges = filtering(trial_inverse, "laplacian")
print(trial_edges[1][1])


## 空の領域

問い：しきい値9で重心はどう表示されるか。

予想：(0,0)と未定義のどちらかを理由付きで予想する。

想定結果：面積0、重心None（未定義）。

説明する問い：空の領域を左上の物体と区別する理由は何か。


In [ ]:
trial_features = features(binary(deepcopy(IMAGE), 9))
print(trial_features)


## テンプレートの回転

問い：90度回転後、元の位置のSSDは0か。

予想：回転前後の画素対応を書いて予想する。

想定結果：回転後[[3,1],[4,2]]、元の位置のSSD=10。最良位置も全探索で確認する。

説明する問い：同じ形でも画素の対応が変わることを説明する。


In [ ]:
trial_template = rotate(deepcopy(TEMPLATE))
trial_rotation = score(window(SCENE, 1, 2, 2, 2), trial_template)
print(trial_template, trial_rotation)
print(match(SCENE, trial_template))


## 明るさの変更

問い：探索画像へ10を加えると最良位置はどうなるか。

予想：SSDとZNCCで同じ結果になるかを予想する。

想定結果：元の位置のSSD=400。ZNCCは1で最良(1,2)。SSDは別の位置を選ぶ。

説明する問い：平均を引く操作の効果と、回転には対応しない理由を説明する。


In [ ]:
trial_bright = tone(deepcopy(SCENE), 10)
trial_ssd = match(trial_bright, TEMPLATE)
trial_zncc = match(trial_bright, TEMPLATE, "zncc")
print(trial_ssd["best"], trial_zncc["best"])
print(score(window(trial_bright, 1, 2, 2, 2), TEMPLATE))


In [ ]:
assert baseline_result == demo()
assert IMAGE[2][2] == 9 and TEMPLATE == [[1,2],[3,4]]
print("基準入力・結果を保持しました")


## 元に戻す・困ったとき

コピーしたNotebookを閉じ、元ファイルから再開します。NameErrorなら上から全セル実行、Jupyterが起動しない場合は既存venvとカーネルを確認してください。入力は有限数の長方形配列、最大16×16です。Noneはエラーではなく未定義。実用性能・Forms採点はこの実験では確認しません。


## 振り返りと復帰

入力・行った処理・結果・変更前との差を、自分の言葉で説明してください。観察セルは毎回基準のコピーから計算するため、同じセルを再実行しても前の実験の続きにはなりません。NameErrorや実行順が不明な場合はカーネル再起動→全セル実行。元の配布ファイルを残し、変更版は別名保存します。

Python版は`python image_recognition.py --explain all`で同じ計算表を表示します。各テーマは`--explain binary`のように指定できます。従来のコマンドと固定期待値も維持しています。
